# Dual-Encoder Evaluation: Temporal vs Tabular Component Analysis

**Date:** 2026-04-02  
**Objective:** Evaluate what each encoder component (temporal vs tabular) is actually good for

## Background

The current model uses **simple concatenation** (`torch.cat([temporal, tabular])`) rather than learned cross-modal fusion. This evaluation separates the 128-dim temporal and 128-dim tabular embeddings into distinct rankers to understand their individual strengths.

In [ ]:
import pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport numpy as npfrom pathlib import Pathimport os# Set stylesns.set_style("whitegrid")plt.rcParams['figure.figsize'] = (14, 8)plt.rcParams['font.size'] = 11# Load results - try multiple path strategies# Strategy 1: Absolute path from workspace rootWORKSPACE_ROOT = Path('/home/redbear/Projects/LiquidSearcher')RESULTS_DIR = WORKSPACE_ROOT / "results" / "retrieval_separate"# Strategy 2: If not found, try relative to current directoryif not (RESULTS_DIR / "metrics" / "evaluation_summary.csv").exists():    RESULTS_DIR = Path("../results/retrieval_separate")# Strategy 3: Try one more relative pathif not (RESULTS_DIR / "metrics" / "evaluation_summary.csv").exists():    RESULTS_DIR = Path("results/retrieval_separate")print(f"Loading results from: {RESULTS_DIR.absolute()}")summary_df = pd.read_csv(RESULTS_DIR / "metrics/evaluation_summary.csv")overall_df = pd.read_csv(RESULTS_DIR / "metrics/retrieval_metrics_overall.csv", index_col=0)print("Loaded evaluation data for:")print(f"  - {len(summary_df)} ranker × reference combinations")print(f"  - {summary_df['ranker'].nunique()} rankers: {list(summary_df['ranker'].unique())}")print(f"  - {summary_df['reference'].nunique()} references: {list(summary_df['reference'].unique())}")

## 1. Overall Performance Summary

Comparing all 4 rankers averaged across all 6 ground truth references:

In [ ]:
# Display overall metrics
print("Overall Performance (averaged across all 6 references):")
print("=" * 70)
display(overall_df.T)

# Create bar plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Recall@10
overall_df.loc['Recall@10'].plot(kind='bar', ax=axes[0], color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[0].set_title('Recall@10 (Overall Average)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Recall@10')
axes[0].set_ylim(0, max(overall_df.loc['Recall@10']) * 1.2)
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(overall_df.loc['Recall@10']):
    axes[0].text(i, v + 0.0001, f'{v:.4f}', ha='center', va='bottom', fontsize=10)

# nDCG@10
overall_df.loc['nDCG@10'].plot(kind='bar', ax=axes[1], color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[1].set_title('nDCG@10 (Overall Average)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('nDCG@10')
axes[1].set_ylim(0, max(overall_df.loc['nDCG@10']) * 1.1)
axes[1].tick_params(axis='x', rotation=45)
for i, v in enumerate(overall_df.loc['nDCG@10']):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "report/over_summary.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Finding: tabular_rerank achieves the best overall performance (nDCG@10: 0.617)")

## 2. Performance by Ground Truth Reference

Each reference tests a different aspect of similarity:

In [ ]:
# Pivot for heatmap
pivot_recall = summary_df.pivot(index='ranker', columns='reference', values='Recall@10_mean')
pivot_ndcg = summary_df.pivot(index='ranker', columns='reference', values='nDCG@10_mean')

# Create heatmaps
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Recall@10 heatmap
sns.heatmap(pivot_recall, annot=True, fmt='.4f', cmap='YlOrRd', ax=axes[0], cbar_kws={'label': 'Recall@10'})
axes[0].set_title('Recall@10 by Ranker and Reference', fontsize=14, fontweight='bold')
axes[0].set_xlabel('')

# nDCG@10 heatmap
sns.heatmap(pivot_ndcg, annot=True, fmt='.3f', cmap='YlGnBu', ax=axes[1], cbar_kws={'label': 'nDCG@10'})
axes[1].set_title('nDCG@10 by Ranker and Reference', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Ground Truth Reference', fontsize=12)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "report/heatmaps.png", dpi=150, bbox_inches='tight')
plt.show()

## 3. Ranker Performance Analysis

### 3.1 Temporal-Only (128-dim)

**Best at:**
- **TurnoverChar** (nDCG@10: 0.389) - capturing trading activity patterns
- **LiquidityChar** (nDCG@10: 0.474) - liquidity percentile similarity

**Analysis:** The temporal encoder captures price/volume dynamics well but struggles with fundamental similarity.

In [ ]:
# Show temporal-only performance
temporal_perf = summary_df[summary_df['ranker'] == 'temporal_only'].sort_values('nDCG@10_mean', ascending=False)
print("Temporal-Only Performance (sorted by nDCG@10):")
print("=" * 70)
for _, row in temporal_perf.iterrows():
    print(f"{row['reference']:20s} | Recall@10: {row['Recall@10_mean']:.4f} | nDCG@10: {row['nDCG@10_mean']:.3f} | Threshold: {row['threshold_used']:.2f}")

### 3.2 Tabular-Only (128-dim)

**Best at:**
- **SectorSimilarity** (nDCG@10: 0.839) - GICS code matching
- **FundamentalsSim** (nDCG@10: 1.000) - market cap similarity

**Analysis:** The tabular encoder excels at fundamental/sector matching but lacks temporal dynamics.

In [ ]:
# Show tabular-only performance
tabular_perf = summary_df[summary_df['ranker'] == 'tabular_only'].sort_values('nDCG@10_mean', ascending=False)
print("Tabular-Only Performance (sorted by nDCG@10):")
print("=" * 70)
for _, row in tabular_perf.iterrows():
    print(f"{row['reference']:20s} | Recall@10: {row['Recall@10_mean']:.4f} | nDCG@10: {row['nDCG@10_mean']:.3f} | Threshold: {row['threshold_used']:.2f}")

### 3.3 Joint (256-dim Concatenation)

**Performance:**
- Middle-of-the-road on most references
- Does NOT significantly outperform individual encoders

**Analysis:** Simple concatenation without learned fusion fails to leverage cross-modal information effectively. The model aligns modalities during training (InfoNCE) but doesn't learn to combine them optimally.

In [ ]:
# Compare joint vs components
comparison_data = []
for ref in summary_df['reference'].unique():
    ref_data = summary_df[summary_df['reference'] == ref]
    temporal_ndcg = ref_data[ref_data['ranker'] == 'temporal_only']['nDCG@10_mean'].values[0]
    tabular_ndcg = ref_data[ref_data['ranker'] == 'tabular_only']['nDCG@10_mean'].values[0]
    joint_ndcg = ref_data[ref_data['ranker'] == 'joint']['nDCG@10_mean'].values[0]
    best_component = max(temporal_ndcg, tabular_ndcg)
    improvement = ((joint_ndcg - best_component) / best_component * 100) if best_component > 0 else 0
    comparison_data.append({
        'reference': ref,
        'temporal': temporal_ndcg,
        'tabular': tabular_ndcg,
        'joint': joint_ndcg,
        'best_component': best_component,
        'joint_vs_best_pct': improvement
    })

comparison_df = pd.DataFrame(comparison_data)
print("Joint vs Best Component Comparison:")
print("=" * 90)
print(f"{'Reference':20s} | {'Temporal':>8s} | {'Tabular':>8s} | {'Joint':>8s} | {'Improvement':>12s}")
print("-" * 90)
for _, row in comparison_df.iterrows():
    improvement_str = f"{row['joint_vs_best_pct']:+.1f}%" if abs(row['joint_vs_best_pct']) > 0.1 else "~"
    print(f"{row['reference']:20s} | {row['temporal']:8.3f} | {row['tabular']:8.3f} | {row['joint']:8.3f} | {improvement_str:>12s}")

print("\nConclusion: Joint (concatenation) does NOT significantly outperform individual components")

### 3.4 Tabular + Liquidity Rerank

**Best at:**
- **LiquidityUplift** (nDCG@10: 0.754) - Dramatically outperforms all others
- **ReturnSimilarity** (nDCG@10: 0.482) - Best for return correlation

**How it works:**
1. Tabular encoder retrieves top-50 candidates by fundamental/sector similarity
2. Liquidity score reranks within the shortlist

**Analysis:** The two-stage approach (fundamental filtering + liquidity reranking) works better than end-to-end learned retrieval for liquidity-aware recommendations.

In [ ]:
# Compare tabular_rerank vs others
print("Tabular + Liquidity Rerank vs Others:")
print("=" * 90)

for ref in summary_df['reference'].unique():
    ref_data = summary_df[summary_df['reference'] == ref].sort_values('nDCG@10_mean', ascending=False)
    rerank_ndcg = ref_data[ref_data['ranker'] == 'tabular_rerank']['nDCG@10_mean'].values[0]
    best_other = ref_data[ref_data['ranker'] != 'tabular_rerank']['nDCG@10_mean'].max()
    improvement = ((rerank_ndcg - best_other) / best_other * 100) if best_other > 0 else 0
    
    rank = list(ref_data['ranker']).index('tabular_rerank') + 1
    print(f"{ref:20s} | Rank: {rank}/4 | nDCG: {rerank_ndcg:.3f} | vs Best: {improvement:+.1f}%")

print("\nKey Finding: Reranker wins on LiquidityUplift (+150.6%) and ReturnSimilarity (+7.4%)")

## 4. Detailed Comparison by Reference

### 4.1 LiquidityUplift (Primary Objective)

In [ ]:
# LiquidityUplift detailed
liq_data = summary_df[summary_df['reference'] == 'LiquidityUplift'].sort_values('nDCG@10_mean', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d62728' if r == 'tabular_rerank' else '#1f77b4' for r in liq_data['ranker']]
bars = ax.bar(liq_data['ranker'], liq_data['nDCG@10_mean'], color=colors)
ax.set_title('LiquidityUplift: nDCG@10 by Ranker', fontsize=14, fontweight='bold')
ax.set_ylabel('nDCG@10')
ax.set_ylim(0, 0.85)

for bar, val in zip(bars, liq_data['nDCG@10_mean']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "report/liquidity_uplift_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nThe tabular_rerank approach achieves 2.5x better nDCG@10 than the base rankers!")

### 4.2 ReturnSimilarity (Temporal Dynamics)

In [ ]:
# ReturnSimilarity detailed
ret_data = summary_df[summary_df['reference'] == 'ReturnSimilarity'].sort_values('nDCG@10_mean', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ca02c' if r == 'joint' else '#1f77b4' for r in ret_data['ranker']]
bars = ax.bar(ret_data['ranker'], ret_data['nDCG@10_mean'], color=colors)
ax.set_title('ReturnSimilarity: nDCG@10 by Ranker (120-day Return Correlation)', fontsize=14, fontweight='bold')
ax.set_ylabel('nDCG@10')
ax.set_ylim(0, 0.55)

for bar, val in zip(bars, ret_data['nDCG@10_mean']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.3f}', ha='center', fontsize=11)

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "report/return_similarity_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nJoint (concatenation) performs best for return correlation, but only marginally.")

### 4.3 SectorSimilarity (Fundamental Matching)

In [ ]:
# SectorSimilarity detailed
sec_data = summary_df[summary_df['reference'] == 'SectorSimilarity'].sort_values('nDCG@10_mean', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#ff7f0e' if r == 'tabular_only' else '#1f77b4' for r in sec_data['ranker']]
bars = ax.bar(sec_data['ranker'], sec_data['nDCG@10_mean'], color=colors)
ax.set_title('SectorSimilarity: nDCG@10 by Ranker (GICS Code Matching)', fontsize=14, fontweight='bold')
ax.set_ylabel('nDCG@10')
ax.set_ylim(0.75, 0.85)

for bar, val in zip(bars, sec_data['nDCG@10_mean']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.003, f'{val:.3f}', ha='center', fontsize=11)

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "report/sector_similarity_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nTabular encoder excels at sector matching - as expected from GICS features in tabular input.")

## 5. Key Findings & Implications

### 5.1 Architecture Limitation: Simple Concatenation

The model uses `torch.cat([temporal_emb, tabular_emb])` without:
- Cross-modal attention
- Learned fusion layers
- Gating mechanisms

**Result:** The 256-dim joint embedding performs no better than individual 128-dim components. The model aligns modalities during training (InfoNCE loss) but never learns to *combine* them effectively.

### 5.2 Component Specialization

| Component | Best For | Why |
|-----------|----------|-----|
| **Temporal** | TurnoverChar, LiquidityChar | Captures price/volume dynamics |
| **Tabular** | SectorSimilarity, FundamentalsSim | GICS codes, market cap in tabular features |
| **Joint** | ReturnSimilarity | Slight advantage for correlation |
| **Rerank** | LiquidityUplift | Two-stage: tabular filter + liquidity rerank |

### 5.3 SHAP Value Mystery Solved

Previous SHAP analysis showed temporal features dominated importance. **Why?**
- The model never learned cross-modal fusion
- At inference, concatenation treats both modalities equally, but training only aligned them
- Temporal features may have more variance or better gradient flow

### 5.4 Recommendations

1. **For immediate liquidity improvement:** Use `tabular_rerank` (top-50 tabular + liquidity rerank)

2. **For architecture redesign:** Consider:
   - Cross-attention between temporal and tabular features
   - Gated fusion: `fused = gate ⊙ temporal + (1-gate) ⊙ tabular`
   - Task-specific heads for different reference types

3. **For training:** Add auxiliary losses that explicitly predict:
   - Sector classification from joint embedding
   - Liquidity uplift from joint embedding

4. **Feature engineering:** The tabular encoder works well for fundamentals - ensure GICS and market cap features are high quality

## 6. Threshold Analysis

Thresholds used for binary relevance labeling:

In [ ]:
# Show thresholds
threshold_df = summary_df[['reference', 'threshold_used']].drop_duplicates()
print("Thresholds Used for Binary Relevance (75th percentile unless specified):")
print("=" * 70)
for _, row in threshold_df.iterrows():
    ref = row['reference']
    thresh = row['threshold_used']
    if ref == 'LiquidityUplift':
        print(f"{ref:20s} | Threshold: {thresh:.1f} (>0 = positive uplift)")
    elif ref == 'SectorSimilarity':
        print(f"{ref:20s} | Threshold: {thresh:.1f} (0=ggroup match, 0.5=sector match)")
    elif ref == 'FundamentalsSim':
        print(f"{ref:20s} | Threshold: {thresh:.1f} (market cap log difference)")
    else:
        print(f"{ref:20s} | Threshold: {thresh:.4f} (75th percentile)")

print("\nNote: All references except LiquidityUplift use 75th percentile threshold for binary relevance.")

## Appendix: Raw Data Summary

Complete evaluation results:

In [ ]:
# Show complete summary table
print("Complete Evaluation Summary:")
print("=" * 100)
display(summary_df.round(4))

# Save formatted version
summary_df.to_csv(RESULTS_DIR / "report/evaluation_summary_formatted.csv", index=False)
print(f"\nSaved formatted summary to: {RESULTS_DIR / 'report/evaluation_summary_formatted.csv'}")

---

**Report Generated:** 2026-04-02  
**Evaluation Period:** 2019-01-01 to 2019-12-31  
**Queries:** 50 random samples  
**Candidates:** ~3,153 symbols per query  

**Files Generated:**
- `results/retrieval_separate/metrics/evaluation_summary.csv` - Main results
- `results/retrieval_separate/report/` - This notebook and figures
- `results/retrieval_separate/retrieval/figures/` - Comparison plots